# LangGraph Chatbot with Gemini & Memory

A clean, structured implementation of a single-node chatbot using **LangGraph**, **Gemini (gemini-2.5-flash)**, and conversation history preservation via **MemorySaver**.

### 1. Import Dependencies

In [1]:
import os
from typing import Annotated, TypedDict
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

### 2. Configure Environment and API Keys

In [2]:
# Set your Gemini API key
# Best practice: Load this from a .env file or environment variable
# os.environ["GEMINI_API_KEY"] = "your-api-key-here"

### 3. Define Graph State

In [3]:
class ChatState(TypedDict):
    # Annotated list with add_messages automatically appends new messages
    messages: Annotated[list[BaseMessage], add_messages]

### 4. Initialize LLM and Chat Node

In [4]:
# Initialize the Google Gemini model
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

def chatnode(state: ChatState):
    # Call LLM with the list of previous messages
    response = llm.invoke(state["messages"])
    # Return the response to update graph state
    return {"messages": [response]}

### 5. Build and Compile Graph with Memory

In [5]:
# Initialize StateGraph builder
builder = StateGraph(ChatState)

# Add the chatnode node
builder.add_node("chatnode", chatnode)

# Define routing
builder.add_edge(START, "chatnode")
builder.add_edge("chatnode", END)

# Add an in-memory checkpointer for conversation memory
memory = MemorySaver()

# Compile the workflow
chatbot = builder.compile(checkpointer=memory)

### 6. Test the Chatbot (First Query)
Provide a configuration containing a `thread_id` to enable memory preservation.

In [6]:
# Conversation config with thread identifier
config = {"configurable": {"thread_id": "session-1"}}

# First query
initial_state = {
    "messages": [HumanMessage(content="what is capital of India")]
}

result = chatbot.invoke(initial_state, config)

# Print response
print("AI:", result["messages"][-1].content)

AI: The capital of India is **New Delhi**.


### 7. Test Memory (Follow-up Query)
By sending a follow-up under the same `thread_id`, the model will reference previous context.

In [7]:
# Follow-up query
follow_up_state = {
    "messages": [HumanMessage(content="What is its main language?")]
}

result = chatbot.invoke(follow_up_state, config)

# Print response
print("AI:", result["messages"][-1].content)

AI: India is a country with immense linguistic diversity, and it doesn't have a single "national language" designated by its constitution.

However, there are two languages that hold official status at the Union government level and are widely used:

1.  **Hindi:** It is the **official language of the Union Government of India** and is the most widely spoken language in terms of native speakers and overall usage, particularly across the northern and central parts of the country.
2.  **English:** It also holds the status of an **additional official language** alongside Hindi. It is extensively used in government, business, higher education, and serves as a lingua franca among people from different linguistic regions, especially in the south and northeast.

Beyond these, the Indian Constitution recognizes **22 "scheduled languages,"** and many states have their own official languages, reflecting the country's rich linguistic tapestry. So, while Hindi is the most spoken and official langu

### 8. Interactive Chat Loop
Run this cell to start an interactive, real-time chat with your LangGraph chatbot. Type `exit`, `quit`, or `q` to stop.

In [8]:
# Real-time interactive chat loop
config = {"configurable": {"thread_id": "interactive-session-1"}}

print("Chatbot started! Type 'exit', 'quit', or 'q' to end the conversation.\n")

while True:
    user_input = input("You: ")
    if user_input.strip().lower() in ["exit", "quit", "q"]:
        print("Goodbye!")
        break
    
    if not user_input.strip():
        continue
        
    # Invoke the chatbot with the user input
    state = {
        "messages": [HumanMessage(content=user_input)]
    }
    result = chatbot.invoke(state, config)
    
    # Print the last message from the assistant
    print(f"AI: {result['messages'][-1].content}\n")

Chatbot started! Type 'exit', 'quit', or 'q' to end the conversation.

AI: Hi there! How can I help you today?

AI: Delhi is a bit unique!

*   **Delhi** itself is a city and a **National Capital Territory (NCT)** of India.
*   **New Delhi** is the official capital of India, and it is located within the larger metropolitan area of Delhi.

So, while Delhi is the territory, **New Delhi** is the specific capital city within it.

Goodbye!
